# InfraNova AI - Kaggle Training Pipeline

This notebook is specifically configured to run the InfraNova-AI training pipeline on Kaggle using free T4x2 or P100 GPUs.

## ⚠️ Prerequisites
Before running this, make sure you have added your datasets to this notebook via the right-hand panel (**Add Input**). Based on your Kaggle workspace, you should add:
- `InfraNova_AI_full` (Your dataset/splits)
- `InfraNova Checkpoint` (Your latest checkpoints to resume from)

Ensure your Accelerator is set to **GPU** in the right-hand panel.

---
## 1. Setup Environment & Clone Repository

In [ ]:
import os
import shutil

# Clean up previous runs if restarting the cell
if os.path.exists('/kaggle/working/InfraNova-AI'):
    shutil.rmtree('/kaggle/working/InfraNova-AI')

# Clone the repository
!git clone https://github.com/Soham-1009/InfraNova-AI.git /kaggle/working/InfraNova-AI

os.chdir('/kaggle/working/InfraNova-AI')

# Install dependencies
!pip install -q albumentations lpips torchmetrics pyyaml tifffile rasterio earthengine-api geemap imagecodecs

import torch
if torch.cuda.is_available():
    print(f'\n✅ GPU active: {torch.cuda.get_device_name(0)}')
else:
    print('\n❌ NO GPU DETECTED - Change Accelerator in Session Options to GPU!')

---
## 2. Link Datasets & Checkpoints
Kaggle mounts datasets as read-only at `/kaggle/input/`. We will link them into our working directory.

In [ ]:
import os

os.chdir('/kaggle/working/InfraNova-AI')

# 1. Link Dataset
DATASET_PATH = '/kaggle/input/datasets/sohamdeshpande10/infranova-ai-full'

os.makedirs('data/landsat9', exist_ok=True)

# If your dataset is a zip file, unzip it. If it's already a folder, create a symlink.
if os.path.exists(f'{DATASET_PATH}/splits.zip'):
    !unzip -q {DATASET_PATH}/splits.zip -d data/landsat9/
    print("Dataset unzipped successfully!")
elif os.path.exists(f'{DATASET_PATH}/splits'):
    !ln -s {DATASET_PATH}/splits data/landsat9/splits
    print("Dataset linked successfully!")
else:
    print(f"⚠️ Could not find splits in {DATASET_PATH}. Please check the exact path in the right-hand panel.")

# 2. Link Checkpoints (so it can resume training)
CHECKPOINT_PATH = '/kaggle/input/datasets/sohamdeshpande10/infranova-checkpoint'
os.makedirs('outputs/models/best', exist_ok=True)
os.makedirs('outputs/models/final', exist_ok=True)

if os.path.exists(CHECKPOINT_PATH):
    !cp -r {CHECKPOINT_PATH}/* outputs/models/best/
    print("Checkpoints copied successfully!")
else:
    print(f"⚠️ Could not find checkpoints in {CHECKPOINT_PATH}.")


---
## 3. Verify Configuration & Start Training!
The codebase is currently configured to train for 500 epochs in `configs/config.yaml`.

In [ ]:
os.chdir('/kaggle/working/InfraNova-AI')

# Check if the config reflects 500 epochs
!grep 'epochs:' configs/config.yaml

# Start training!
!PYTHONPATH=. python src/training/train_landsat.py

---
## 4. Save Your Work (Important!)
Kaggle will wipe the `/kaggle/working` directory when the session ends. Run this cell to zip up your new checkpoints and visualizations so you can download them from the 'Output' section on the right-hand panel.

In [ ]:
os.chdir('/kaggle/working')

print("Zipping checkpoints and logs...")
!zip -r latest_run_results.zip InfraNova-AI/outputs/models InfraNova-AI/outputs/visualizations InfraNova-AI/logs
print("Done! You can now download latest_run_results.zip from the Output panel on the right.")